In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [19]:
DATA_DIR = Path(r"C:\Users\Jordi\Desktop\Trading\quant_lab\data\raw")
COLUMNS = ["date", "open", "high", "low", "close", "volume", "interest"]


resultats = []
for x in sorted(DATA_DIR.glob("*_RAD.CSV")):
    code = x.stem.removesuffix("_RAD")
    actif = pd.read_csv(x, names=COLUMNS)
    actif["date"] = pd.to_datetime(actif["date"], format="%m/%d/%Y")

    rendement_max = actif['close'].pct_change().abs().max()

    first_date = actif["date"].min()
    last_date = actif["date"].max()
    jours_ouvres = len(pd.bdate_range(first_date, last_date))
    pct_missing = round(100 * (1 - len(actif) / jours_ouvres), 1)
    max_gap_days = actif["date"].diff().max().days

    if (pct_missing < 10) and (max_gap_days < 30) :

        resultats.append({
            "code": code,
            "rendement_max": rendement_max,
            "first_date": first_date,
            "last_date": last_date,
            "n_rows": len(actif),
            "max_gap_days": max_gap_days,
            "pct_missing": pct_missing,
            "min_close": actif["close"].min(),
        })
    else:
        print(f"Actif exclus {code}")

inventaire = pd.DataFrame(resultats).set_index("code")
inventaire

Actif exclus AD
Actif exclus AP


ValueError: Neither `start` nor `end` can be NaT

In [18]:

codes = inventaire.index.drop("BG")
series = []

for code in codes:
    actif = pd.read_csv(DATA_DIR / f"{code}_REV.CSV", names=COLUMNS)
    actif["date"] = pd.to_datetime(actif["date"], format="%m/%d/%Y")
    series.append(actif.set_index("date")["close"].pct_change().rename(code))

rendements = pd.concat(series, axis=1, sort=True)

print(rendements.shape)

verif = pd.DataFrame({
    "max_abs_ret": rendements.abs().max(),
    "date": rendements.abs().idxmax(),
}).sort_values("max_abs_ret", ascending=False)

verif.head(10)



(14829, 71)


,max_abs_ret,date
CC,inf,2009-07-31
LB,inf,2018-10-23
DA,inf,1997-12-16
ZK,459.575540,2026-11-03
ZB,279.000000,2021-11-03
ZI,175.275229,2026-04-21
ZH,98.500000,2011-04-21
JO,60.000000,1977-09-27
CT,57.000000,2009-04-06
ZU,57.000000,2017-07-24
